In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())


2.2.2


In [2]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo
from joblib import Parallel, delayed

# Paths
data_path_ml = "/pool/data/ERA5/E5/ml/an/1D/"
data_path_pl = "/pool/data/ERA5/E5/pl/an/1D/"
hour_path_sf = "/pool/data/ERA5/E5/sf/an/1H/"
day_path_sf  = "/pool/data/ERA5/E5/sf/an/1D/"

scratch_path = "/scratch/u/u301827/paris/"
final_path   = "/work/uc1275/u301827/02_MSE/paris/raw/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

# Paris coordinates
PARIS_LON = 2.35
PARIS_LAT = 48.86
lon_min = lon_max = PARIS_LON
lat_min = lat_max = PARIS_LAT


def compute_monthly_tasmax_paris(year, month, hour_path_sf, final_path):
    """
    Compute daily tasmax from hourly ERA5 T2M, selecting only the Paris grid cell.
    Aggregates all daily tasmax into a single monthly NetCDF file.

    Parameters
    ----------
    year : int
    month : int
    hour_path_sf : str
        Path to hourly ERA5 surface data (1H).
    final_path : str
        Output folder for monthly tasmax file.

    Returns
    -------
    str : path to monthly tasmax file
    """

    os.makedirs(final_path, exist_ok=True)
    cdo = Cdo()

    # Output file
    monthly_file = os.path.join(final_path, f"tasmax_{year}-{month:02d}_paris.nc")

    # List of daily T2M files
    daily_t2m_files = []
    month_str = f"{year}-{month:02d}"

    # Loop over all possible days in the month
    # Let pandas handle month length (28–31)
    date_index = pd.date_range(start=f"{month_str}-01",
                               end=f"{month_str}-28")  # last 3 days might fail safely
    # Extend until month changes
    while date_index[-1].month == month:
        date_index = date_index.append(pd.DatetimeIndex([date_index[-1] + pd.Timedelta(days=1)]))

    # Storage for daily datasets
    daily_datasets = []

    for day in date_index:
        if day.month != month:
            break

        date_str = day.strftime("%Y-%m-%d")

        t2m_file = f"{hour_path_sf}167/E5sf00_1H_{date_str}_167.grb"
        if not os.path.exists(t2m_file):
            print(f"Skipping missing: {t2m_file}")
            continue

        # Temporary files
        reg_file   = f"/tmp/reg_t2m_{date_str}.nc"
        paris_file = f"/tmp/paris_t2m_{date_str}.nc"

        # Step 1: convert GRIB → netCDF (regular grid)
        cdo.setgridtype("regular", input=t2m_file, output=reg_file, options="-f nc --eccodes")

        # Step 2: select Paris grid cell (nearest grid point)
        cdo.remapnn(f"lon={PARIS_LON}_lat={PARIS_LAT}", input=reg_file, output=paris_file)

        # Step 3: compute tasmax from hourly data
        ds = xr.open_dataset(paris_file)
        tasmax_val = ds["2t"].max(dim="time")
        tasmax_val = tasmax_val.rename("tasmax")

        # Assign the daily date
        tasmax_val = tasmax_val.expand_dims(time=[pd.to_datetime(date_str)])

        daily_datasets.append(tasmax_val)

        ds.close()
        os.remove(reg_file)
        os.remove(paris_file)

    # Combine all daily tasmax into 1 month
    if len(daily_datasets) == 0:
        print(f"No valid tasmax files for {year}-{month:02d}")
        return None

    ds_month = xr.concat(daily_datasets, dim="time")
    ds_month.to_netcdf(monthly_file)

    return monthly_file


In [3]:
start_date = "1940-01-01"
end_date   = "2024-12-31"

# Create all year-month pairs
months = pd.date_range(start=start_date, end=end_date, freq="MS")

def run_all_months(hour_path_sf, final_path, n_jobs=50):
    """Parallel runner for compute_monthly_tasmax_paris."""
    
    tasks = [
        delayed(compute_monthly_tasmax_paris)(
            date.year, date.month, hour_path_sf, final_path
        )
        for date in months
    ]

    # Run in parallel
    Parallel(n_jobs=n_jobs, verbose = 10, backend="multiprocessing")(tasks)


In [4]:
#run_all_months(hour_path_sf, final_path, n_jobs=50)


In [4]:
def process_era5_paris(year, month, var_num, var,
                       levels="ml", level="137"):
    """
    Process ERA5 daily data selecting ONLY the Paris grid cell.
    
    Notes:
    - ERA5 daily (1D) files in the pool are already aggregated from hourly data,
      so no daily aggregation is necessary here.
    - This function extracts the Paris grid cell and applies vertical level
      selection where required (model or pressure levels).
    """

    cdo = Cdo()
    date_str = f"{year}-{month:02d}"

    # Output folders
    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)

    out_file = os.path.join(var_final, f"{var}_{date_str}_paris.nc")

    # ---------------------------------------------------------
    # Determine which ERA5 path to use
    # ---------------------------------------------------------
    if levels == "ml":
        data_path = data_path_ml
    elif levels == "pl":
        data_path = data_path_pl
    else:
        data_path = day_path_sf   # surface daily fields

    var_file = f"{data_path}{var_num}/E5{levels}00_1D_{date_str}_{var_num}.grb"
    if not os.path.exists(var_file):
        print(f"Missing: {var_file}")
        return

    # Step 1: Convert GRIB → NetCDF + regular grid
    reg_file = os.path.join(var_scratch, f"reg_{var}_{date_str}.nc")
    cdo.setgridtype("regular", input=var_file, output=reg_file, options="-f nc --eccodes")

    # Step 2: Select only the Paris grid cell
    paris_file = os.path.join(var_scratch, f"paris_{var}_{date_str}.nc")
    # Step 2: select Paris grid cell (nearest grid point)
    cdo.remapnn(f"lon={PARIS_LON}_lat={PARIS_LAT}", input=reg_file, output=paris_file)

    # ---------------------------------------------------------
    # Step 3: Apply vertical level logic
    # ---------------------------------------------------------

    # Model levels (ML)
    if levels == "ml":
        cdo.sellevel(level, input=paris_file, output=out_file)
        os.remove(paris_file)

    # Pressure levels (PL)
    elif levels == "pl":
        ds = xr.open_dataset(paris_file)
        ds_sel = ds.sel(plev=int(level))
        ds_sel["time"] = pd.to_datetime(ds_sel["time"].values)
        ds_sel.to_netcdf(out_file)
        ds.close()
        os.remove(paris_file)

    # Surface fields (SF)
    else:
        # Surface fields (SF)
        import shutil
        shutil.move(paris_file, out_file)

    # Clean up temp file
    os.remove(reg_file)

    return out_file


In [ ]:
# === Variable mapping ===
era5_vars = {
    134: "sp",
    133: "q",   # Specific humidity @ ML level 137
    167: "2t",  # 2m temperature (surface field)
    168: "2d",  # 2m dewpoint temperature (surface field)
    39: "swvl1",
    159: "blh",
    
    130: "t",   # Temperature @ 500 hPa (pressure level)
    129: "z",   # Geopotential @ 500 hPa (pressure level)
}

# === Date range (monthly) ===
start_date = "1940-01-01"
end_date   = "2024-12-31"
months = pd.date_range(start_date, end_date, freq="MS")  # monthly start dates

# === Wrapper function for correct level handling ===
def run_process(date, var_num, var):

    # Pressure-level variables (500 hPa)
    if var_num in [130, 129]:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="pl",
            level="50000"
        )

    # Model-level variable (q at ML 137)
    elif var_num == 133:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="ml",
            level="137"
        )

    # Surface variable (2m temperature)
    elif var_num in [39, 134, 167, 159, 168, 232]:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="sf"
        )

    else:
        print(f"Variable {var_num} not supported.")
        return None

# === Parallel execution ===
from joblib import Parallel, delayed

n_jobs = 50

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_process)(month, var_num, var)
    for var_num, var in era5_vars.items()
    for month in months
)

print("Processing completed.")


[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.
[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:   11.5s
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:   11.7s
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:   11.8s
[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:   13.5s
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:   14.0s
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:   14.1s
[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:   15.5s
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:   16.1s
[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:   17.3s
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:   18.0s
[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:   19.2s
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:   19.9s
[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:   21.1s
[Parallel(n_jobs=50)]: Done 292 tasks      | elapsed:   22.0s
[Parallel(n_jobs=50)]: Done 321 tasks      | elapsed:  

In [7]:
import os
import pandas as pd
from joblib import Parallel, delayed

# === Example variable mapping ===
# We'll just test one variable first, e.g., 2m temperature (surface)
era5_vars = {
    167: "2t",  # 2m temperature, surface (no levels)
}

# === Date range for testing ===
start_date = "2021-06-01"
end_date   = "2021-06-01"
months = pd.date_range(start=start_date, end=end_date, freq="MS")

# === Wrapper to call process_era5_paris correctly ===
def run_process(month, var_num, var):
    """
    Calls process_era5_paris for a single month and variable,
    automatically setting levels depending on the variable.
    """
    # Pressure-level variables
    if var_num in [130, 129]:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="pl", level="50000"
        )

    # Model-level variables
    elif var_num == 133:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="ml", level="137"
        )

    # Surface variables (2m temperature)
    elif var_num == 167:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="sf"
        )

    else:
        print(f"Variable {var_num} not configured.")
        return None

# === Test execution for a single month ===
for var_num, var in era5_vars.items():
    for month in months:
        out_file = run_process(month, var_num, var)
        if out_file:
            print(f"Processed {var} for {month.strftime('%Y-%m')}: {out_file}")
        else:
            print(f"Failed to process {var} for {month.strftime('%Y-%m')}")


Processed 2t for 2021-06: /work/uc1275/u301827/02_MSE/paris/raw/2t/2t_2021-06_paris.nc


## 